In [7]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

train_ds = keras.utils.image_dataset_from_directory(
    directory="/home/karan/DL2.0/Datasets/PetImages/train",
    labels="inferred", label_mode="int",
    batch_size=64, image_size=(256, 256),
    color_mode="rgb"  
)
val_ds = keras.utils.image_dataset_from_directory(
    directory="/home/karan/DL2.0/Datasets/PetImages/validation",
    labels="inferred", label_mode="int",
    batch_size=64, image_size=(256, 256),
    color_mode="rgb"
)
test_ds = keras.utils.image_dataset_from_directory(
    directory="/home/karan/DL2.0/Datasets/PetImages/test",
    labels="inferred", label_mode="int",
    batch_size=64, image_size=(256, 256),
    color_mode="rgb"
)

AUTOTUNE = tf.data.AUTOTUNE
def process(img, label):
    return tf.cast(img / 255.0, tf.float32), label

train_ds = train_ds.map(process).cache().prefetch(AUTOTUNE)
val_ds   = val_ds.map(process).cache().prefetch(AUTOTUNE)
test_ds  = test_ds.map(process).cache().prefetch(AUTOTUNE)

# Model
model = Sequential([
    keras.Input(shape=(256, 256, 3)),
    Conv2D(32, (3,3), activation="relu", padding="same"),
    MaxPooling2D(2, 2),
    Conv2D(64, (3,3), activation="relu", padding="same"),
    MaxPooling2D(2, 2),
    Conv2D(128, (3,3), activation="relu", padding="same"),
    MaxPooling2D(2, 2),
    MaxPooling2D(2, 2),  # extra pool — flatten size kam hoga
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

# Train
with tf.device('/GPU:0'):
    history = model.fit(train_ds, validation_data=val_ds, epochs=10)

Found 17456 files belonging to 2 classes.
Found 3739 files belonging to 2 classes.
Found 3741 files belonging to 2 classes.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,296,001 (16.39 MB)

 Trainable params: 4,296,001 (16.39 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
 20/273 ━━━━━━━━━━━━━━━━━━━━ 39s 157ms/step - accuracy: 0.4918 - loss: 0.7507

InvalidArgumentError: Graph execution error:

Detected at node decode_image/DecodeImage defined at (most recent call last):
<stack traces unavailable>
Detected at node decode_image/DecodeImage defined at (most recent call last):
<stack traces unavailable>
2 root error(s) found.
  (0) INVALID_ARGUMENT:  Number of channels inherent in the image must be 1, 3 or 4, was 2
	 [[{{node decode_image/DecodeImage}}]]
	 [[IteratorGetNext]]
	 [[IteratorGetNext/_4]]
  (1) INVALID_ARGUMENT:  Number of channels inherent in the image must be 1, 3 or 4, was 2
	 [[{{node decode_image/DecodeImage}}]]
	 [[IteratorGetNext]]
0 successful operations.
0 derived errors ignored. [Op:__inference_multi_step_on_iterator_5462]